<div class="alert alert-block alert-info">

## Imports & Connection
</div>

In [1]:
import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import datetime as datetime

from scipy import stats
from pyhive import presto
from IPython.display import display
from datetime import datetime, timedelta

import warnings
warnings.filterwarnings('ignore')

In [2]:
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 300)

In [3]:
## Connection
connection = presto.connect(
        host='presto-gateway.serving.data.production.internal',
        port=80,
        protocol='http',
        catalog='hive',
        username='manoj.ravirajan@rapido.bike'
)

In [4]:
print('Notebook dir - ', os.getcwd())
local_extract = '/Users/E2074/local-datasets/customer/patience'
print('Data dir - ', local_extract)

Notebook dir -  /Users/E2074/analytics/customer/patience
Data dir -  /Users/E2074/local-datasets/customer/patience


<div class="alert alert-block alert-info">
    
## Datasets & Parameter
</div>

In [5]:
## Parameter 
start_date = '20241001'
end_date = '20241031'
city = ['Delhi', 'Bangalore']
service = ['Link', 'Auto', 'CabEconomy']

In [6]:
def higher_gross_orders_cluster_hex(start_date, end_date, city, service):
    
    city_str = ", ".join(f"'{c}'" for c in city)
    service_str = ", ".join(f"'{s}'" for s in service)
    
    base_query = f"""

    SELECT
        *
    FROM
        (
        SELECT
            date_range,
            city_name,
            pickup_cluster,
            pickup_location_hex_8 pickup_loc_hex8,
            SUM(orders) OVER(PARTITION BY city_name) total_orders,
            orders,
            TRY(orders*100.00/SUM(orders) OVER(PARTITION BY city_name)) "gross contribution%",
            ROW_NUMBER() OVER(PARTITION BY city_name ORDER BY orders DESC) "hex_demand_rank"
        FROM
            (
            SELECT 
                city_name,
                pickup_cluster, 
                pickup_location_hex_8,
                COUNT(DISTINCT order_id) orders,
                MIN(yyyymmdd) || ' to ' || MAX(yyyymmdd) date_range
            FROM  
                orders.order_logs_fact fact
            WHERE 
                yyyymmdd >= '{start_date}'
                AND yyyymmdd <= '{end_date}'
                AND city_name IN ({city_str})
                AND service_obj_service_name IN ({service_str})
            GROUP BY 1,2,3
            )
        )
    WHERE 
        hex_demand_rank <= 15
    ORDER BY 6 DESC

    """
    
    df_base = pd.read_sql(base_query, connection)

    return df_base

In [7]:
df_higher_gross_orders_cluster_hex = higher_gross_orders_cluster_hex(start_date, end_date, city, service)
display(df_higher_gross_orders_cluster_hex)


list_top_hex8 = df_higher_gross_orders_cluster_hex['pickup_loc_hex8'].tolist()

,date_range,city_name,pickup_cluster,pickup_loc_hex8,total_orders,orders,gross contribution%,hex_demand_rank
0,20241001 to 20241031,Bangalore,Majestic,8860145b55fffff,23580911,223297,0.95,1
1,20241001 to 20241031,Bangalore,St Johns Signal,88618925c7fffff,23580911,210134,0.89,2
2,20241001 to 20241031,Bangalore,Tavarakere,88618925c3fffff,23580911,158623,0.67,3
3,20241001 to 20241031,Bangalore,Tavarakere,8861892589fffff,23580911,158401,0.67,4
4,20241001 to 20241031,Bangalore,Ejipura,88618925e9fffff,23580911,136214,0.58,5
5,20241001 to 20241031,Bangalore,BTM,88618925cbfffff,23580911,135500,0.57,6
6,20241001 to 20241031,Bangalore,Madiwala,88618925c1fffff,23580911,131439,0.56,7
7,20241001 to 20241031,Bangalore,Green Glen Layout,8861892543fffff,23580911,130168,0.55,8
8,20241001 to 20241031,Bangalore,Metro - MG Road,8861892e9bfffff,23580911,129645,0.55,9
9,20241001 to 20241031,Bangalore,Munnekollal Sankara Eye Hospital,88618920bdfffff,23580911,125426,0.53,10


In [8]:
def month_order_data(start_date, end_date, city, service, list_top_hex8):
    
    city_str = ", ".join(f"'{c}'" for c in city)
    service_str = ", ".join(f"'{s}'" for s in service)
    hex8 = ", ".join(f"'{h}'" for h in list_top_hex8)

    
    base_query = f"""

    with olf as(
        select 
            yyyymmdd, 
            city_name,
            service_obj_service_name service_name,
            service_detail_id,
            pickup_cluster,
            pickup_location_hex_8 pickup_loc_hex8,
            customer_id,
            order_id, 
            order_status, 
            modified_order_status, 
            spd_fraud_flag,
            rider_accepted_epoch,
            order_requested_epoch,
            customer_cancelled_epoch,
            dropped_epoch,
            expired_epoch,
            quarter_hour,
            hhmmss,
            channel_host,
            CASE 
                WHEN modified_order_status IN ('OCARA', 'COBRA', 'COBRM') THEN (customer_cancelled_epoch / 1000 - order_requested_epoch / 1000) 
            END AS time_taken_to_cancel,
            
            CASE 
                WHEN order_status = 'dropped' OR modified_order_status IN ('OCARA', 'aborted') THEN (rider_accepted_epoch / 1000 - order_requested_epoch / 1000) 
            END AS time_taken_to_accept,
            
            CASE 
                WHEN CAST(SUBSTR(hhmmss, 1, 2) AS INTEGER) BETWEEN 8 AND 11 THEN '2. Morning_Peak'
                WHEN CAST(SUBSTR(hhmmss, 1, 2) AS INTEGER) BETWEEN 12 AND 16 THEN '3. Afternoon'
                WHEN CAST(SUBSTR(hhmmss, 1, 2) AS INTEGER) BETWEEN 17 AND 21 THEN '4. Evening_Peak'
                WHEN CAST(SUBSTR(hhmmss, 1, 2) AS INTEGER) BETWEEN 0 AND 7 THEN '1. Rest_Morning'
                ELSE '5. Rest_Evening'
            END AS time_bucket,
            CASE 
                WHEN distance_final_distance >= 0 AND distance_final_distance < 5 THEN '1. Short'
                WHEN distance_final_distance >= 5 AND distance_final_distance < 9 THEN '2. Medium'
                WHEN distance_final_distance >= 9 THEN '3. Long'
            END AS lm_bucket
            
        from 
            orders.order_logs_fact fact
        where 
            yyyymmdd >= '{start_date}'
                AND yyyymmdd <= '{end_date}'
                AND city_name IN ({city_str})
                AND service_obj_service_name IN ({service_str})
                AND pickup_location_hex_8 IN ({hex8})
            ),
            
            
        segment AS (
        
        SELECT
            customer_id,
            taxi_retention_segments retention_segment,
            taxi_regularity_segment regularity_segment
        FROM
            datasets.iallocator_customer_segments
        WHERE 
            run_date = date_format(date_parse('{end_date}', '%Y%m%d'),'%Y-%m-%d')
        
        ),
            
        merge AS (
        
        select 
            olf.*,
            segment.retention_segment,
            segment.regularity_segment
        from 
            olf
        left join 
            segment
            on olf.customer_id = segment.customer_id
        )
        
        select
            yyyymmdd,
            city_name,
            service_name,
            service_detail_id,
            pickup_cluster,
            pickup_loc_hex8,
            customer_id,
            retention_segment,
            regularity_segment,
            order_id,
            order_status,
            modified_order_status,
            spd_fraud_flag,
            rider_accepted_epoch,
            order_requested_epoch,
            customer_cancelled_epoch,
            dropped_epoch,
            expired_epoch,
            -- quarter_hour,
            -- hhmmss,
            time_bucket,
            lm_bucket,
            time_taken_to_cancel,
            time_taken_to_accept,
            channel_host
        from 
             merge

    """
    
    df_base = pd.read_sql(base_query, connection)
    df_base.to_parquet(local_extract + '/patience_oct_month_data.parquet', index=False)

    return df_base

df_base = month_order_data(start_date, end_date, city, service, list_top_hex8)

In [10]:
df_base = pd.read_parquet(local_extract + '/patience_oct_month_data.parquet')
print(df_base.shape)
df_base.head()

,yyyymmdd,city_name,service_name,service_detail_id,pickup_cluster,pickup_loc_hex8,customer_id,retention_segment,regularity_segment,order_id,order_status,modified_order_status,spd_fraud_flag,rider_accepted_epoch,order_requested_epoch,customer_cancelled_epoch,dropped_epoch,expired_epoch,time_bucket,lm_bucket,time_taken_to_cancel,time_taken_to_accept,channel_host
0,20241016,Delhi,Link,5da4660028af187d8d52cc3c,MG Road Metro,883da11a99fffff,62f430e0c30fe205218d15e7,ELITE,DAILY,670f9d162999400fc4eae4c1,customerCancelled,OCARA,None,1.729077e+12,1729076502475,1.729077e+12,NaN,NaN,3. Afternoon,1. Short,39.0,8.0,android
1,20241016,Bangalore,Auto,5c53562fceb6fc9241980547,Tavarakere,88618925c3fffff,6389b2577d8d602ff2940094,ELITE,DAILY,670fa8c454154a2b6d159b7d,dropped,dropped,False,1.729079e+12,1729079492373,NaN,1.729080e+12,NaN,4. Evening_Peak,1. Short,NaN,7.0,ios
2,20241016,Bangalore,Auto,5c53562fceb6fc9241980547,BTM,88618925cbfffff,641bfc40f89689a9f9663283,ELITE,DAILY,670fbb0c8249d16f9e8c1cc4,dropped,dropped,False,1.729084e+12,1729084172809,NaN,1.729088e+12,NaN,4. Evening_Peak,3. Long,NaN,38.0,android
3,20241016,Delhi,Auto,5fc8c40bf4f0a70007d7cfbc,Rajiv Chowk Metro,883da11409fffff,642bd2d4beab3373e4d5237e,GOLD,BI_WEEKLY,670fbb9735a50c35d33e7f46,dropped,dropped,False,1.729084e+12,1729084311057,NaN,1.729088e+12,NaN,4. Evening_Peak,3. Long,NaN,156.0,android
4,20241016,Delhi,Auto,5fc8c40bf4f0a70007d7cfbc,T Udyog Vihar Phase 3,883da111a9fffff,624258586e78d15828fad956,ELITE,DAILY,670fc7dbad4cfe03099bc258,dropped,dropped,False,1.729087e+12,1729087451860,NaN,1.729089e+12,NaN,4. Evening_Peak,2. Medium,NaN,4.0,android


<div class="alert alert-block alert-success">
    
### Analysis
</div>

### Benchmark TTA & TTC

In [27]:
df_benchmark_tta_ttc = df_base\
.groupby(['city_name', 'pickup_loc_hex8', 'service_name', 'time_bucket', 'lm_bucket'])\
.agg(
    # orders = ('order_id', 'nunique'),
    bmark_avg_tta=('time_taken_to_accept', 'mean'),
    bmark_p50_tta=('time_taken_to_accept', lambda x: np.percentile(x.dropna(), 50) if len(x.dropna()) > 0 else np.nan),
    # bmark_p75_tta=('time_taken_to_accept', lambda x: np.percentile(x.dropna(), 75) if len(x.dropna()) > 0 else np.nan),
    # bmark_p90_tta=('time_taken_to_accept', lambda x: np.percentile(x.dropna(), 90) if len(x.dropna()) > 0 else np.nan),
    # bmark_p95_tta=('time_taken_to_accept', lambda x: np.percentile(x.dropna(), 95) if len(x.dropna()) > 0 else np.nan),
    bmark_avg_ttc=('time_taken_to_cancel', 'mean'),
    bmark_p50_ttc=('time_taken_to_cancel', lambda x: np.percentile(x.dropna(), 50) if len(x.dropna()) > 0 else np.nan),
    # bmark_p75_ttc=('time_taken_to_cancel', lambda x: np.percentile(x.dropna(), 75) if len(x.dropna()) > 0 else np.nan),
    # bmark_p90_ttc=('time_taken_to_cancel', lambda x: np.percentile(x.dropna(), 90) if len(x.dropna()) > 0 else np.nan),
    # bmark_p95_ttc=('time_taken_to_cancel', lambda x: np.percentile(x.dropna(), 95) if len(x.dropna()) > 0 else np.nan)
    )\
.reset_index()


df_benchmark_tta_ttc.head()

,city_name,pickup_loc_hex8,service_name,time_bucket,lm_bucket,bmark_avg_tta,bmark_p50_tta,bmark_avg_ttc,bmark_p50_ttc
0,Bangalore,8860145b55fffff,Auto,1. Rest_Morning,1. Short,99.687466,21.0,234.450968,151.0
1,Bangalore,8860145b55fffff,Auto,1. Rest_Morning,2. Medium,83.045213,11.0,228.247492,154.0
2,Bangalore,8860145b55fffff,Auto,1. Rest_Morning,3. Long,70.901464,6.0,234.315757,147.0
3,Bangalore,8860145b55fffff,Auto,2. Morning_Peak,1. Short,84.821517,15.0,261.439210,163.0
4,Bangalore,8860145b55fffff,Auto,2. Morning_Peak,2. Medium,66.759740,7.0,254.987100,168.0


### Segment

In [34]:
df_segment_tta_ttc = df_base\
.groupby(['city_name', 'pickup_loc_hex8', 'service_name', 'time_bucket', 'lm_bucket', 'retention_segment'])\
.agg(
    gross_orders = ('order_id', 'nunique'),
    avg_tta=('time_taken_to_accept', 'mean'),
    p50_tta=('time_taken_to_accept', lambda x: np.percentile(x.dropna(), 50) if len(x.dropna()) > 0 else np.nan),
    # p75_tta=('time_taken_to_accept', lambda x: np.percentile(x.dropna(), 75) if len(x.dropna()) > 0 else np.nan),
    # p90_tta=('time_taken_to_accept', lambda x: np.percentile(x.dropna(), 90) if len(x.dropna()) > 0 else np.nan),
    # p95_tta=('time_taken_to_accept', lambda x: np.percentile(x.dropna(), 95) if len(x.dropna()) > 0 else np.nan),
    avg_ttc=('time_taken_to_cancel', 'mean'),
    p50_ttc=('time_taken_to_cancel', lambda x: np.percentile(x.dropna(), 50) if len(x.dropna()) > 0 else np.nan),
    # p75_ttc=('time_taken_to_cancel', lambda x: np.percentile(x.dropna(), 75) if len(x.dropna()) > 0 else np.nan),
    # p90_ttc=('time_taken_to_cancel', lambda x: np.percentile(x.dropna(), 90) if len(x.dropna()) > 0 else np.nan),
    # p95_ttc=('time_taken_to_cancel', lambda x: np.percentile(x.dropna(), 95) if len(x.dropna()) > 0 else np.nan)
    )\
.reset_index()


df_segment_tta_ttc.head()

,city_name,pickup_loc_hex8,service_name,time_bucket,lm_bucket,retention_segment,gross_orders,avg_tta,p50_tta,avg_ttc,p50_ttc
0,Bangalore,8860145b55fffff,Auto,1. Rest_Morning,1. Short,DORMANT,1142,109.684116,23.0,234.904685,151.0
1,Bangalore,8860145b55fffff,Auto,1. Rest_Morning,1. Short,ELITE,888,77.701068,15.0,229.163814,130.0
2,Bangalore,8860145b55fffff,Auto,1. Rest_Morning,1. Short,GOLD,1398,92.673243,21.0,208.761983,146.0
3,Bangalore,8860145b55fffff,Auto,1. Rest_Morning,1. Short,HH,499,99.884244,22.0,261.592391,202.5
4,Bangalore,8860145b55fffff,Auto,1. Rest_Morning,1. Short,INACTIVE,217,286.650000,129.0,330.557047,190.0


### Merge

In [36]:
df_merge1 = pd.merge(df_segment_tta_ttc, df_benchmark_tta_ttc,
                     how = 'left',
                     on = ['city_name', 'pickup_loc_hex8', 'service_name', 'time_bucket', 'lm_bucket'])

df_merge2 = pd.merge(df_merge1, df_higher_gross_orders_cluster_hex[['pickup_cluster', 'pickup_loc_hex8', 'hex_demand_rank']],
                     how = 'inner',
                     on = ['pickup_loc_hex8'])

In [37]:
df_merge2

,city_name,pickup_loc_hex8,service_name,time_bucket,lm_bucket,retention_segment,gross_orders,avg_tta,p50_tta,avg_ttc,p50_ttc,bmark_avg_tta,bmark_p50_tta,bmark_avg_ttc,bmark_p50_ttc,pickup_cluster,hex_demand_rank
0,Bangalore,8860145b55fffff,Auto,1. Rest_Morning,1. Short,DORMANT,1142,109.684116,23.0,234.904685,151.0,99.687466,21.0,234.450968,151.0,Majestic,1
1,Bangalore,8860145b55fffff,Auto,1. Rest_Morning,1. Short,ELITE,888,77.701068,15.0,229.163814,130.0,99.687466,21.0,234.450968,151.0,Majestic,1
2,Bangalore,8860145b55fffff,Auto,1. Rest_Morning,1. Short,GOLD,1398,92.673243,21.0,208.761983,146.0,99.687466,21.0,234.450968,151.0,Majestic,1
3,Bangalore,8860145b55fffff,Auto,1. Rest_Morning,1. Short,HH,499,99.884244,22.0,261.592391,202.5,99.687466,21.0,234.450968,151.0,Majestic,1
4,Bangalore,8860145b55fffff,Auto,1. Rest_Morning,1. Short,INACTIVE,217,286.650000,129.0,330.557047,190.0,99.687466,21.0,234.450968,151.0,Majestic,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11802,Delhi,883da11ad5fffff,Link,5. Rest_Evening,3. Long,HH,14,340.285714,167.0,1446.714286,1170.0,125.492611,23.0,578.738462,197.0,IFFCO Forest,15
11803,Delhi,883da11ad5fffff,Link,5. Rest_Evening,3. Long,INACTIVE,5,98.250000,55.5,518.600000,164.0,125.492611,23.0,578.738462,197.0,IFFCO Forest,15
11804,Delhi,883da11ad5fffff,Link,5. Rest_Evening,3. Long,PLATINUM,43,70.256410,17.0,465.000000,200.0,125.492611,23.0,578.738462,197.0,IFFCO Forest,15
11805,Delhi,883da11ad5fffff,Link,5. Rest_Evening,3. Long,SILVER,24,101.500000,26.5,169.142857,167.0,125.492611,23.0,578.738462,197.0,IFFCO Forest,15


In [41]:
df_final = df_merge2[['city_name','pickup_loc_hex8','service_name','time_bucket','lm_bucket','retention_segment','gross_orders',
           'bmark_avg_tta','avg_tta','bmark_p50_tta','p50_tta','bmark_avg_ttc','avg_ttc',
           'bmark_p50_ttc','p50_ttc','pickup_cluster','hex_demand_rank']]

In [39]:
df_final.to_clipboard(index=False)